# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [ ]:
# Setup
%pip install -r requirements.txt
import os, json
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  
)
MODEL = "llama3.2:3b"



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [2]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise support assistant"},
        {"role": "user", "content": "What is the capital of France?"}
    ],
    temperature=0.1
)

print(response.choices[0].message.content)
print(response.usage)

print("=== Temperature 0.1 ===")
for i in range(3):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a concise support assistant"},
            {"role": "user", "content": "What is the capital of France?"}
        ],
        temperature=0.1
    )
    print(f"Run {i+1}: {response.choices[0].message.content}")

print("\n=== Temperature 1.0 ===")
for i in range(3):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a concise support assistant"},
            {"role": "user", "content": "What is the capital of France?"}
        ],
        temperature=1.0
    )
    print(f"Run {i+1}: {response.choices[0].message.content}")

The capital of France is Paris.
CompletionUsage(completion_tokens=8, prompt_tokens=38, total_tokens=46, completion_tokens_details=None, prompt_tokens_details=None)
=== Temperature 0.1 ===
Run 1: The capital of France is Paris.
Run 2: The capital of France is Paris.
Run 3: The capital of France is Paris.

=== Temperature 1.0 ===
Run 1: The capital of France is Paris.
Run 2: The capital of France is Paris.
Run 3: Paris.


**What changed, and why?**

At **temperature 0.1**, all three runs returned the identical phrasing: *"The capital of France is Paris."* This is expected — low temperature makes the model strongly favor the highest-probability token at each step, producing near-deterministic output.

At **temperature 1.0**, all three runs returned just *"Paris."* — slightly different from the low-temperature runs, but still consistent across all three. This suggests that for this very simple factual question, the probability distribution is so heavily concentrated on a short, direct answer that even a higher temperature doesn't introduce much variation. With a small local model like `llama3.2:3b`, the effect of temperature may also be less pronounced than with larger hosted models, since the model's confidence in short factual answers tends to dominate the sampling regardless of temperature.

In general, temperature controls how "flat" or "sharp" the next-token probability distribution is before sampling: low temperature sharpens it (favoring the most likely tokens, more deterministic and repetitive output), while high temperature flattens it (giving lower-probability tokens a real chance of being picked, increasing variability — more noticeable on open-ended or creative prompts than on short factual ones like this).

## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [ ]:
with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

FEW_SHOT_EXAMPLES = """Example 1:
Message: "I can't log in, it says invalid password but I'm sure it's correct."
Label: bug

Example 2:
Message: "My card was charged but I never got an order confirmation."
Label: billing

Example 3:
Message: "Could you add the ability to export data to Excel?"
Label: feature_request
"""

def classify(text, style):
    if style == "zero_shot":
        prompt = f"""Classify the following support ticket into exactly one of these labels: billing, bug, feature_request, other.
Respond with ONLY the label, nothing else.

Message: "{text}"
Label:"""

    elif style == "few_shot":
        prompt = f"""Classify the following support ticket into exactly one of these labels: billing, bug, feature_request, other.
Respond with ONLY the label, nothing else.

{FEW_SHOT_EXAMPLES}
Now classify this message:
Message: "{text}"
Label:"""

    elif style == "cot":
        prompt = f"""Classify the following support ticket into exactly one of these labels: billing, bug, feature_request, other.

First, briefly reason about what the message is asking for (1-2 sentences).
Then, on a new line, write "Label: " followed by ONLY the label.

Message: "{text}"
"""
    else:
        raise ValueError("unknown style")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a concise support ticket classifier."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content.strip()


def extract_label(raw_output):
    """Pull the label from the last 'Label:' line; fallback to scanning text."""
    for line in reversed(raw_output.strip().splitlines()):
        if "label:" in line.lower():
            after = line.lower().split("label:")[-1].strip()
            for label in LABELS:
                if label in after:
                    return label
    text = raw_output.lower()
    for label in LABELS:
        if label in text:
            return label
    return "unparsed"

results = []
for ticket in tickets:
    row = {"id": ticket["id"], "text": ticket["text"]}
    for style in ["zero_shot", "few_shot", "cot"]:
        raw = classify(ticket["text"], style)
        row[style] = raw
        row[f"{style}_label"] = extract_label(raw)
    results.append(row)

print(f"{'ID':<4}{'Zero-shot':<18}{'Few-shot':<18}{'CoT (extracted)':<18}")
for r in results:
    print(f"{r['id']:<4}{r['zero_shot_label']:<18}{r['few_shot_label']:<18}{r['cot_label']:<18}")

print("\n=== Full CoT outputs ===")
for r in results:
    print(f"Ticket {r['id']}: {r['cot']}")

ID  Zero-shot         Few-shot          CoT (extracted)   
1   billing           billing           bug               
2   bug               bug               bug               
3   feature_request   feature_request   feature_request   
4   bug               bug               bug               
5   bug               bug               bug               
6   billing           billing           billing           
7   feature_request   feature_request   feature_request   
8   feature_request   feature_request   feature_request   

=== Full CoT outputs ===
Ticket 1: The user is reporting an error with their billing and requesting a refund for an incorrect charge. They are seeking assistance to resolve the issue.

Label: bug
Ticket 2: This support ticket is asking for assistance with an issue where the export button on the reports page is not functioning correctly and throwing a server error. The user is seeking help to resolve this problem.

Label: bug
Ticket 3: The user is requesting a chan

## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [4]:
def classify_structured(text):
    prompt = f"""Classify the following support ticket.

Respond with ONLY a JSON object (no other text, no markdown) in exactly this format:
{{"label": "<one of: billing, bug, feature_request, other>", "confidence": <number between 0 and 1>, "reason": "<short explanation>"}}

For example:
{{"label": "bug", "confidence": 0.95, "reason": "App crashes on startup"}}

Message: "{text}"
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a concise support ticket classifier. Respond with valid JSON only, using real values, not the schema placeholders."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1
    )
    raw = response.choices[0].message.content.strip()

    if raw.startswith("```"):
        raw = raw.strip("`")
        raw = raw.replace("json\n", "", 1).replace("json", "", 1).strip()

    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        return {"label": "other", "confidence": 0.0, "reason": "PARSE_ERROR: invalid JSON", "raw": raw}

    label = str(data.get("label", "")).strip().lower()
    if label not in LABELS:
        data["label"] = "other"
        data["reason"] = str(data.get("reason", "")) + " [INVALID_LABEL corrected to 'other']"
    else:
        data["label"] = label

    conf = data.get("confidence", 0.0)
    if not isinstance(conf, (int, float)) or not (0 <= conf <= 1):
        data["confidence"] = 0.5
        data["reason"] = str(data.get("reason", "")) + " [INVALID_CONFIDENCE corrected to 0.5]"

    return data


structured_results = []
for ticket in tickets:
    result = classify_structured(ticket["text"])
    structured_results.append({"id": ticket["id"], **result})
    print(f"Ticket {ticket['id']}: {result}")

Ticket 1: {'label': 'billing', 'confidence': 0.9, 'reason': 'Subscription payment issue'}
Ticket 2: {'label': 'bug', 'confidence': 0.98, 'reason': 'Export functionality not working as expected'}
Ticket 3: {'label': 'feature_request', 'confidence': 0.9, 'reason': 'User requested feature for improved user experience'}
Ticket 4: {'label': 'other', 'confidence': 0.8, 'reason': 'Password recovery instructions not clearly provided'}
Ticket 5: {'label': 'bug', 'confidence': 0.98, 'reason': 'Crash occurs immediately after launch, suggesting a recent update or system issue'}
Ticket 6: {'label': 'billing', 'confidence': 0.9, 'reason': 'Request for account-related information'}
Ticket 7: {'label': 'feature_request', 'confidence': 0.9, 'reason': 'User requests additional report format option'}
Ticket 8: {'label': 'feature_request', 'confidence': 0.05, 'reason': 'User is expressing appreciation for a feature that was recently implemented'}


**Local vs hosted: did the small local model produce valid JSON as reliably?**

> A hosted-vs-local comparison could not be completed as planned, since the Gemini API key consistently returned quota errors across multiple keys/accounts, even on the first call. As a result, this entire lab — including the structured-output task — was run against the local Ollama model (`llama3.2:3b`).
>
> For the local model, JSON reliability turned out to depend heavily on prompt phrasing rather than just temperature. An initial prompt that described the schema using a `"label": "billing|bug|feature_request|other"` placeholder caused the model to copy that literal string into the `label` field (e.g. `"billing|bug"`), which failed validation on 6 of 8 tickets and was caught by the `INVALID_LABEL` fallback. After rewriting the prompt to spell out the allowed values in plain language and include one fully worked example JSON object, the model returned 8/8 valid, parseable JSON objects with a correct `label` (from the allowed set) and `confidence` in [0, 1] — no fallback corrections were needed.
>
> So while `llama3.2:3b` did **not** produce valid JSON reliably on the first attempt, it became fully reliable once the prompt was made unambiguous with a concrete example — suggesting the small local model is capable of structured output, but is more sensitive to exact prompt wording than a larger hosted model would likely be.
